# Lesson 4: Running Algorithms

**Duration:** ~10 minutes  
**Module:** 5 - GDS with Python  
**Dataset:** Movies (Users, Movies)

## What You'll Learn

- Algorithm call syntax in Python
- The four execution modes and when to use each
- Chaining algorithms together
- Streaming properties from projections
- Including database properties (names, titles) in results

## Setup: Connect and Create Projection

In [1]:
import os
import pandas as pd
from IPython.display import display
from graphdatascience import GraphDataScience
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

uri = os.getenv('NEO4J_URI')
username = os.getenv('NEO4J_USERNAME')
password = os.getenv('NEO4J_PASSWORD')

# Create GDS client
gds = GraphDataScience(uri, auth=(username, password))

print(f"Connected to GDS version: {gds.version()}")

Connected to GDS version: 2.25.0


In [2]:
# Clean up any existing projections
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")
print("Ready to start!")

Ready to start!


In [3]:
# Create our working projection: Users and Movies connected by RATED
G, _ = gds.graph.project(
    "user-movie-graph",
    {
        "User": {},
        "Movie": {
            "properties": {
                "year": {"defaultValue": 1900},
                "imdbRating": {"defaultValue": 0.0}
            }
        }
    },
    {
        "RATED": {
            "orientation": "UNDIRECTED",
            "properties": {
                "rating": {"defaultValue": 3.0}
            }
        }
    }
)

print(f"Projected graph: {G.name()}")
print(f"Nodes: {G.node_count()}")
print(f"Relationships: {G.relationship_count()}")

Projected graph: user-movie-graph
Nodes: 9793
Relationships: 200006


## Algorithm Syntax

The pattern is: `gds[.tier].<algorithm>.<mode>(G, **config)`

```python
# Louvain community detection with stream mode
result = gds.louvain.stream(G)

# Louvain with mutate mode
result = gds.louvain.mutate(G, mutateProperty="community")

# Beta algorithm
result = gds.beta.node2vec.stream(G, embeddingDimension=64)
```

## Execution Modes Compared

| **Mode** | **Returns** | **Side Effect** |
|----------|-------------|----------------|
| `.stream()` | DataFrame with per-node/relationship results | None |
| `.mutate()` | Series with summary statistics | Adds property to projection |
| `.write()` | Series with summary statistics | Writes property to Neo4j |
| `.stats()` | Series with summary statistics | None |

## Stream Mode

Returns results as a DataFrame—perfect for analysis and exploration.

In [4]:
# Stream Louvain community detection results
df = gds.louvain.stream(G)

print("Louvain community detection results:")
print(f"Columns: {df.columns.tolist()}")
display(df.head())

 Louvain:  11%|#         | 10.61/100 [00:00<?, ?%/s]

Louvain community detection results:
Columns: ['nodeId', 'intermediateCommunityIds', 'communityId']


,nodeId,intermediateCommunityIds,communityId
0,0,None,8234
1,1,None,8234
2,2,None,8234
3,3,None,8234
4,4,None,8902


In [5]:
# Work with results using pandas
# Count nodes per community
community_sizes = df.groupby("communityId").size().reset_index(name="size")
print("Largest communities:")
display(community_sizes.nlargest(10, "size"))

Largest communities:


,communityId,size
60,8295,3004
59,8234,2590
62,8649,1600
63,8802,1277
64,8902,751
61,8435,512
0,2794,1
1,2833,1
2,3084,1
3,3148,1


## Mutate Mode

Adds results to the projection (not the database). Useful for chaining algorithms.

In [6]:
# Check current properties
print(f"User properties before: {G.node_properties('User')}")
print(f"Movie properties before: {G.node_properties('Movie')}")

User properties before: []
Movie properties before: ['year', 'imdbRating']


In [7]:
# Mutate: store community in the projection
result = gds.louvain.mutate(
    G,
    mutateProperty="community"
)

print(f"Nodes processed: {result['nodePropertiesWritten']}")
print(f"Communities found: {result['communityCount']}")
print(f"Modularity: {result['modularity']:.4f}")
print(f"Compute time: {result['computeMillis']}ms")

 Louvain:   8%|7         | 7.74/100 [00:00<?, ?%/s]

Nodes processed: 9793
Communities found: 66
Modularity: 0.3051
Compute time: 1047ms


In [8]:
# Property is now available in the projection
print(f"User properties after: {G.node_properties('User')}")
print(f"Movie properties after: {G.node_properties('Movie')}")

User properties after: ['community']
Movie properties after: ['imdbRating', 'community', 'year']


## Write Mode

Writes results directly to Neo4j—useful for persisting results.

In [9]:
# Write community to the database
result = gds.louvain.write(
    G,
    writeProperty="community"
)

print(f"Wrote to {result['nodePropertiesWritten']} nodes")
print(f"Communities found: {result['communityCount']}")

 Louvain:   9%|8         | 8.69/100 [00:00<?, ?%/s]

Wrote to 9793 nodes
Communities found: 66


In [10]:
# Verify by querying the database
df = gds.run_cypher("""
    MATCH (u:User)
    WHERE u.community IS NOT NULL
    RETURN u.community AS community, count(*) AS userCount
    ORDER BY userCount DESC
    LIMIT 5
""")
print("Largest user communities (from database):")
display(df)

Largest user communities (from database):


,community,userCount
0,8295,203
1,9034,171
2,8627,147
3,8516,121
4,8649,24


## Stats Mode

Returns only statistics—useful for modifying the algorithm's configuration before persisting results.

We can run stats mode and just return the dataframe:

In [12]:
result = gds.louvain.stats(G)

print(result)

 Louvain:  10%|9         | 9.64/100 [00:00<?, ?%/s]

modularity                                                        0.305931
modularities                      [0.2925135062763613, 0.3059307686785416]
ranLevels                                                                2
communityCount                                                          67
communityDistribution    {'p1': 1, 'max': 3480, 'p5': 1, 'p90': 253, 'p...
preProcessingMillis                                                      0
computeMillis                                                         1276
postProcessingMillis                                                     1
configuration            {'maxIterations': 10, 'seedProperty': None, 'c...
Name: 0, dtype: object


Or we can grab columns from the dataframe to specify results:

In [13]:
# Get statistics only
result = gds.louvain.stats(G)

print(f"Community count: {result['communityCount']}")
print(f"Modularity: {result['modularity']:.4f}")
print(f"Compute time: {result['computeMillis']}ms")

 Louvain:  10%|9         | 9.67/100 [00:00<?, ?%/s]

Community count: 68
Modularity: 0.2755
Compute time: 1534ms


## Chaining Algorithms

Use mutate mode to chain algorithms together—each algorithm's output becomes input for the next.

In [14]:
# Add degree centrality to complement community detection
result = gds.degree.mutate(G, mutateProperty="degree")
print(f"Added degree centrality to {result['nodePropertiesWritten']} nodes")

Added degree centrality to 9793 nodes


In [15]:
# Check: both properties now exist in the projection
print(f"User properties: {G.node_properties('User')}")
print(f"Movie properties: {G.node_properties('Movie')}")

User properties: ['degree', 'community']
Movie properties: ['degree', 'imdbRating', 'community', 'year']


In [16]:
# Stream all results together
df = gds.graph.nodeProperties.stream(
    G,
    node_properties=["community", "degree"],
        listNodeLabels=True
)

print("Combined algorithm results:")
display(df.head(10))

Combined algorithm results:


,nodeId,nodeProperty,propertyValue,nodeLabels
0,0,community,8627.0,[Movie]
1,0,degree,33.0,[Movie]
2,1,community,8627.0,[Movie]
3,1,degree,2.0,[Movie]
4,2,community,8627.0,[Movie]
5,2,degree,28.0,[Movie]
6,3,community,8627.0,[Movie]
7,3,degree,28.0,[Movie]
8,4,community,8627.0,[Movie]
9,4,degree,11.0,[Movie]


## Memory Estimation

Estimate algorithm memory requirements before running—useful for large graphs.

In [17]:
# Estimate memory for Louvain
estimate = gds.louvain.mutate.estimate(
    G,
    mutateProperty="community_new"
)

print(f"Required memory: {estimate['requiredMemory']}")
print(f"Node count: {estimate['nodeCount']}")
print(f"Relationship count: {estimate['relationshipCount']}")

Required memory: [618 KiB ... 6249 KiB]
Node count: 9793
Relationship count: 200006


## Summary

You've learned to run algorithms in Python:

| Mode | Returns | Side Effect | Use When |
|------|---------|-------------|----------|
| `.stream()` | DataFrame | None | Exploring results |
| `.mutate()` | Series | Adds to projection | Chaining algorithms |
| `.write()` | Series | Writes to Neo4j | Persisting results |
| `.stats()` | Series | None | Quick statistics |

### Next Lesson

In Lesson 5, you'll put it all together in a complete workflow.

In [18]:
# Clean up
G.drop()
print("Projection dropped. Lesson complete!")

Projection dropped. Lesson complete!
